# 資料前處理(Label encoding、 One hot encoding)
這兩個編碼方式的目的是為了將類別 (categorical)或是文字(text)的資料轉換成數字，而讓程式能夠更好的去理解及運算。
> Label encoding : 把每個類別 mapping 到某個整數，不會增加新欄位

> One hot encoding : 為每個類別新增一個欄位，用 0/1 表示是否

![](images/Encoder.PNG)


### Encoding Categorical features (or label)
![](images/Encoding.PNG)


In [1]:
import pandas as pd
import numpy as np


## 練習二：Keras - label encoder + to_categorical

In [2]:
def to_categorical(y, num_classes=None, dtype='float32'):
    """Converts a class vector (integers) to binary class matrix.
    E.g. for use with categorical_crossentropy.
    # Arguments
        y: class vector to be converted into a matrix
            (integers from 0 to num_classes).
        num_classes: total number of classes.
        dtype: The data type expected by the input, as a string
            (`float32`, `float64`, `int32`...)
    # Returns
        A binary matrix representation of the input. The classes axis
        is placed last.
    # Example
    ```python
    # Consider an array of 5 labels out of a set of 3 classes {0, 1, 2}:
    > labels
    array([0, 2, 1, 2, 0])
    # `to_categorical` converts this into a matrix with as many
    # columns as there are classes. The number of rows
    # stays the same.
    > to_categorical(labels)
    array([[ 1.,  0.,  0.],
           [ 0.,  0.,  1.],
           [ 0.,  1.,  0.],
           [ 0.,  0.,  1.],
           [ 1.,  0.,  0.]], dtype=float32)
    ```
    """
    #將輸入y向量轉換為陣列
    y = np.array(y, dtype='int')
    #獲取陣列的行列大小
    input_shape = y.shape
    if input_shape and input_shape[-1] == 1 and len(input_shape) > 1:
        input_shape = tuple(input_shape[:-1])
    #y變為1維陣列
    y = y.ravel()
    #如果使用者沒有輸入分類個數，則自行計算分類個數
    if not num_classes:
        num_classes = np.max(y) + 1
    n = y.shape[0]
    #生成全為0的n行num_classes列的值全為0的矩陣
    categorical = np.zeros((n, num_classes), dtype=dtype)
    #np.arange(n)得到每個行的位置值，y裡邊則是每個列的位置值
    categorical[np.arange(n), y] = 1
    #進行reshape矯正
    output_shape = input_shape + (num_classes,)
    categorical = np.reshape(categorical, output_shape)
    return categorical

In [3]:
import numpy as np
import pandas as pd
country=['Taiwan','Australia','Ireland','Australia','Ireland','Taiwan']
age=[25,30,45,35,22,36]
salary=[20000,32000,59000,60000,43000,52000]
dic={'Country':country,'Age':age,'Salary':salary}
data=pd.DataFrame(dic)


from sklearn.preprocessing import LabelEncoder
import np_utils

# label encoder 
encoder = LabelEncoder()
encoded_Y = encoder.fit_transform(data['Country'])
print(encoded_Y)
data['Country']=encoded_Y
data

# convert integers to one hot encoding
dummy_y = to_categorical(encoded_Y)  # one-hot encoding [0. 1. 0.] 
dummy_y


[2 0 1 0 1 2]


array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

In [3]:
import numpy as np
import pandas as pd
df = pd.DataFrame({'blood':['A','B','AB','O','B'], 
                   'Y':['high','low','high','mid','mid'],
                   'Z':[np.nan,np.nan,-1196,72,83]})

# 🌟 魔法參數 columns=['blood']：
# 告訴 Pandas：「整張表給你，但請你只幫我把 blood 欄位展開，其他的保留原樣！」
df_final = pd.get_dummies(df, columns=['blood'], dtype=int)

print(df_final)

      Y       Z  blood_A  blood_AB  blood_B  blood_O
0  high     NaN        1         0        0        0
1   low     NaN        0         0        1        0
2  high -1196.0        0         1        0        0
3   mid    72.0        0         0        0        1
4   mid    83.0        0         0        1        0


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
df = pd.DataFrame({'blood':['A','B','AB','O','B'], 
                   'Y':['high','low','high','mid','mid'],
                   'Z':[np.nan,np.nan,-1196,72,83]})


# 1. 建立自動分揀輸送帶 (設定對 'blood' 和 'Y' 欄位做 OneHotEncoding，剩下的欄位 'passthrough' 直接放行)
ct = ColumnTransformer(
    transformers=[
        ("Onehot", OneHotEncoder(sparse_output=False), ['blood', 'Y']) 
    ],
    remainder='passthrough' # 其他沒點到名的欄位 (例如 Z)，直接保留放行！
)

# 🌟 最新版的神級設定：要求它吐出 Pandas DataFrame，而不是 NumPy 陣列！
ct.set_output(transform="pandas")

# 2. 直接把整張 df 丟進去轉換！
df_sklearn_final = ct.fit_transform(df)

print(df_sklearn_final)

ColumnTransformer(remainder='passthrough',
                  transformers=[('Onehot', OneHotEncoder(sparse_output=False),
                                 ['blood', 'Y'])])
   Onehot__blood_A  Onehot__blood_AB  Onehot__blood_B  Onehot__blood_O  \
0              1.0               0.0              0.0              0.0   
1              0.0               0.0              1.0              0.0   
2              0.0               1.0              0.0              0.0   
3              0.0               0.0              0.0              1.0   
4              0.0               0.0              1.0              0.0   

   Onehot__Y_high  Onehot__Y_low  Onehot__Y_mid  remainder__Z  
0             1.0            0.0            0.0           NaN  
1             0.0            1.0            0.0           NaN  
2             1.0            0.0            0.0       -1196.0  
3             0.0            0.0            1.0          72.0  
4             0.0            0.0            1.0          83.0 